# 29 homa fabric

Question: do the data centre mechanisms of the engine of the crate `homa`, priorities with
cutoffs, the shared downlink with overcommitment, incast control and all or nothing scheduling,
behave as the papers of Homa report, in the crate's own simulator of a rack.

The side under test is `aosabi-devtool homa-sim` with its operation `fabric`, which runs one
endpoint of the full form on every host of a `homa::sim::Fabric`, one switch with strict priority
queues and a shared buffer, both roles on every host, and reports every RPC, the counters, the
switch and a trace of every grant decision. The reference side is `tests/harness/homa/fabric.py`,
which holds the workloads W1 and W4 of HomaModule's `util/dist.cc`, the time of an RPC on an idle
fabric, `homa_grant_priority`, the cutoff rule of SIGCOMM 2018 and the percentiles, and
`tests/harness/homa/trace.py`, which checks the grant decisions over the trace. What the notebook
asserts is computed from the answers and the trace of the devtool with the reference. The sender of
each decision, the bytes a host expects and the GRANTs on the wire are read from the rows of the
packets and the RPCs the notebook sent.

The comparison with HomaModule on a real network is the interop half of `29_homa_raw`, stage H4.

The exit number, stage H1 of the plan's section The notebooks: at 80 percent load on a mix of W1
and W4, the P99 slowdown of 100 byte RPCs at most 3 with 8 levels and at most 4 with 2, the P99
of messages under 1 KB with 8 levels at most that with 4, with 4 at most that with 2 and with 2
at most that with 1, and at least 5 times lower than the same engine as a byte stream baseline. A
fan out of 1, 100, 1,000 and 4,000 RPCs with 10 KB responses: 4,000 complete with 0 drops at the
switch with incast control, and drops above a printed count without it. The receiver's downlink
at least 85 percent used with `max_overcommit` 8 and below 70 percent with 1. 0 grants to more
than one message per sender, granted and unreceived bytes never above `max_incoming`, the
scheduled levels of every grant equal to `homa_grant_priority`'s, and computed cutoffs that give
each unscheduled level its share of bytes within 10 percent.

Judged stage: H1 of the Homa track.

In [ ]:
# Papermill overrides this cell.
# The tree under comparison.
aosabi_root = "."

# The seed of the workloads, the arrivals and the samples.
seed = 20260927

# The hosts of the rack of the load runs, and of the larger rack.
hosts = 16
big_hosts = 40

# The load of the exit number, and the loads the table prints.
load = 0.8
loads = "0.1,0.5,0.8"

# The nanoseconds of arrivals of one load run, after a millisecond of
# warm up, and the seeds whose runs the level sweep pools.
duration_ns = 20_000_000
level_seeds = 3

# The fan outs of the incast runs, each of 10 KB responses.
fan_outs = "1,100,1000,4000"

# The load offered in the overcommitment runs.
overcommit_load = 1.0

# The messages one receiver counts before it computes its cutoffs.
cutoff_messages = 100000

# The nanoseconds of arrivals of a traced run.
trace_duration_ns = 5_000_000

# The per command timeout.
timeout_s = 600

# The pinned toolchain the crate is built with.
rust_version = "1.95.0"

## Setup

The cell prints the toolchain and the gates the notebook rests on, every cargo call with
`--locked`. The fabric runs move millions of packets, so this notebook builds the devtool with
the release profile and drives `target/release/aosabi-devtool`, the same source as every other
notebook's debug build. Every batch is written under `tests/notebooks/out/29_homa_fabric/`, which
git ignores, so a run is replayed by hand with the command the notebook runs.

In [ ]:
import json
import math
import os
import random
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import paths, run
from harness.homa import fabric, trace

P = paths(aosabi_root)
OUT = P.out / "29_homa_fabric"
OUT.mkdir(parents=True, exist_ok=True)
DEVTOOL = P.target / "release" / "aosabi-devtool"
CARGO_BIN = Path(os.environ.get("CARGO_HOME") or Path.home() / ".cargo") / "bin"
CARGO = CARGO_BIN / "cargo"
RUSTUP = CARGO_BIN / "rustup"
RUSTFMT = CARGO_BIN / "rustfmt"

GATES = [
    ("cargo build --locked --release -p aosabi-devtool",
     [CARGO, "build", "--locked", "--release", "-p", "aosabi-devtool"]),
    ("cargo clippy --locked --workspace --all-targets -- -D warnings",
     [CARGO, "clippy", "--locked", "--workspace", "--all-targets", "--", "-D", "warnings"]),
    ("rustfmt --check --edition 2021 crates/homa/src/lib.rs",
     [RUSTFMT, "--check", "--edition", "2021", P.root / "crates" / "homa" / "src" / "lib.rs"]),
    ("rustfmt --check --edition 2024 crates/aosabi-devtool/src/main.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi-devtool" / "src" / "main.rs"]),
]
TARGETS = run([RUSTUP, "target", "list", "--installed"], cwd=P.root, timeout_s=60).out.split()
GATE_FAILURES = []
for label, argv in GATES:
    gate = run(argv, cwd=P.root, timeout_s=900, check=False)
    print("gate %-66s exit %d, %.1f s" % (label, gate.returncode, gate.duration_s))
    if gate.returncode != 0:
        GATE_FAILURES.append(label)
        print((gate.err or gate.out).strip()[-2000:])

reported_cargo = run([CARGO, "--version"], cwd=P.root, timeout_s=timeout_s).out.strip()
smoke = json.loads(run([DEVTOOL, "homa-sim", "--smoke"], cwd=P.root, timeout_s=timeout_s).out)
print("repository root:", P.rel(P.root))
print("devtool        :", P.rel(DEVTOOL))
print("batches        :", P.rel(OUT))
print("cargo          :", reported_cargo)
print("targets        :", ", ".join(TARGETS))
print("seam           :", json.dumps(smoke))
print("seed           :", seed)
assert not GATE_FAILURES, GATE_FAILURES
assert rust_version in reported_cargo, reported_cargo
assert smoke == {"subcommand": "homa-sim", "implemented": True, "lands_at": "H0"}, smoke

PANICS = []


def sim(requests, name):
    """Send fabric runs to `homa-sim` in one process and return its answers."""
    path = OUT / ("%s.ndjson" % name)
    path.write_text("".join(json.dumps(r) + "\n" for r in requests), encoding="utf-8")
    result = run(
        [DEVTOOL, "homa-sim", "--requests", path], cwd=P.root, timeout_s=timeout_s, check=False
    )
    answers = [json.loads(line) for line in result.out.splitlines()]
    if result.returncode != 0 or len(answers) != len(requests):
        PANICS.append("%s: exit %d, %d answers to %d requests, %s" % (
            name, result.returncode, len(answers), len(requests), result.err.strip()[-300:]))
    bad = [a for a in answers if "fatal" in a or "config_error" in a]
    assert not bad, "the seam refused runs of %s: %s" % (name, bad[:2])
    return answers

## The fabric and the time of an idle RPC

Every run below is one switch with `hosts` ports of 25 Gbit/s, a propagation of 750 ns each way of a
hop, a shared buffer of 13 MB and eight strict priority queues a port. A host sends one packet at a
time on its link, and the switch stores each packet whole and forwards it once the port to its
receiver is free, highest level first, and drops a packet the buffer has no room for. The link rate
is that of the 40 node CloudLab cluster HomaModule was measured on, the propagation, 1.5
microseconds a way, stands for the software delay of 1.5 microseconds a host adds in the simulations
of SIGCOMM 2018, which the simulator has none of, and the buffer of 13 MB is the notebook's choice
of the shared buffer of a switch of that class. Every host runs the Network profile in all or
nothing scheduling, its rate pacer at 24 Gbit/s with the backlog of HomaModule, and a warm up RPC
between every pair of hosts at time 0, so every host knows the cutoffs of every other before the
measured arrivals start at 1 ms.

The slowdown of an RPC is its time over the time the same RPC takes on the idle fabric, which
`fabric.py` computes from the link rate, the headers and the propagation alone, a request sent back
to back and stored and forwarded at the switch and a response of 0 bytes. The cell times one RPC of
each size on an idle fabric of two hosts, whose time must lie within 2 percent of the reference's,
so the slowdowns below measure queueing and not a difference of the two models.

In [ ]:
RATE = 25_000_000_000
PROPAGATION = 750
FABRIC = {"link_bits_per_s": RATE, "propagation_ns": PROPAGATION, "buffer_bytes": 13_000_000,
          "priority_queues": 8}
PACER = {"bits_per_s": 24_000_000_000}
START_NS = 1_000_000


def request(hosts, rpcs, name, levels=8, cutoffs=None, overrides=None, config=None, extra=None):
    """The fabric run of `rpcs` on `hosts` hosts, with a warm up RPC
    between every pair of hosts at time 0 before them."""
    warm = [[c, s, 0, 8, 0] for c in range(hosts) for s in range(hosts) if c != s]
    req = {
        "id": name, "op": "fabric", "fabric": dict(FABRIC, hosts=hosts),
        "profile": {"name": "network"}, "mode": "all-or-nothing", "levels": levels,
        "config": config or {"client_slots": 1024, "server_slots": 1024, "page_len": 65536,
                             "pages": 2048, "max_gaps": 16},
        "pacer": PACER, "rpcs": warm + rpcs, "max_time_ns": 10_000_000_000,
        "measure_from_ns": START_NS,
    }
    if cutoffs is not None:
        req["cutoffs"] = cutoffs
    if overrides:
        req["overrides"] = overrides
    if extra:
        req.update(extra)
    return req, len(warm)


def slowdowns(answer, rpcs, warm, longest):
    """The slowdowns of the answered RPCs whose request is at most `longest`
    bytes, and the count of the others that did not end answered."""
    out, unanswered = [], 0
    for (client, server, at, length, response), (issued, delivered, done, outcome) in zip(
            rpcs, answer["rpcs"][warm:]):
        if outcome != 1:
            unanswered += 1
            continue
        if length <= longest:
            out.append((done - issued) / fabric.unloaded_rpc_ns(length, response, RATE, PROPAGATION))
    return out, unanswered


SIZES = [1, 100, 1000, 10000, 50000, 300000]
idle = sim([request(2, [[0, 1, START_NS, n, 0]], "idle-%d" % n, config={
    "client_slots": 8, "server_slots": 8, "page_len": 65536, "pages": 64, "max_gaps": 16})[0]
    for n in SIZES], "idle")
IDLE_WORST = 0.0
print("%-10s %-12s %-12s %s" % ("request", "fabric ns", "reference", "ratio"))
for n, answer in zip(SIZES, idle):
    issued, delivered, done, outcome = answer["rpcs"][-1]
    reference = fabric.unloaded_rpc_ns(n, 0, RATE, PROPAGATION)
    ratio = (done - issued) / reference
    IDLE_WORST = max(IDLE_WORST, abs(ratio - 1))
    print("%-10d %-12d %-12d %.4f" % (n, done - issued, reference, ratio))
assert IDLE_WORST <= 0.02, IDLE_WORST

## The workload

W1 and W4 are the tables of `util/dist.cc` of HomaModule at the pinned commit, W1 the distribution
of the small messages of an RPC store and W4 the Hadoop cluster's of the SIGCOMM 2018 paper. The mix
draws each message from one of them with equal chance, a length between two points of the table
uniformly and at most the 1,000,000 bytes a message of Homa holds. Every client host sends requests
of those lengths to a server drawn uniformly among the other hosts, with Poisson arrivals at the
rate that fills `load` of its link with the DATA packets of the requests, headers and framing
included, the definition of load of SIGCOMM 2018 section 5.2.

The cutoffs of each level count are the ones that give each unscheduled level an equal share of the
unscheduled bytes of the mix, the rule of SIGCOMM 2018 section 3.4, computed by `fabric.py` over a
sample of 200,000 lengths, and the scheduled levels are the lower half of the levels, three of eight
as in HomaModule's default.

In [ ]:
MIX = ["w1", "w4"]
sample = fabric.Workload(MIX, seed).draw
LENGTHS = [sample() for _ in range(200_000)]
CUTOFFS = {levels: fabric.equal_share_cutoffs(LENGTHS, levels) for levels in (8, 4, 2, 1)}
under_1k = sum(1 for n in LENGTHS if n < 1000) / len(LENGTHS)
at_100 = sum(1 for n in LENGTHS if n <= 100) / len(LENGTHS)
print("sampled lengths: %d, mean %.0f bytes, median %d, at most 100 bytes %.1f %%, under 1 KB %.1f %%,"
      " above 40,000 bytes %.1f %%" % (
          len(LENGTHS), sum(LENGTHS) / len(LENGTHS), sorted(LENGTHS)[len(LENGTHS) // 2], 100 * at_100,
          100 * under_1k, 100 * sum(1 for n in LENGTHS if n > 40_000) / len(LENGTHS)))
for levels, cut in CUTOFFS.items():
    print("cutoffs of %d levels: %s" % (levels, cut))


def poisson(hosts, load, duration, run_seed, workload=MIX, clients=None):
    return fabric.poisson_rpcs(hosts, load, RATE, duration, fabric.Workload(workload, run_seed), run_seed + 1,
                               start_ns=START_NS, clients=clients)

## Priorities: 8, 4, 2 and 1 levels at 80 percent load

Each level count runs the same arrivals on `hosts` hosts at `load`, once for each of `level_seeds`
seeds, and the slowdowns of the seeds are pooled before the percentile is read, since the
distance between 8 and 4 levels is smaller than what one seed of 20 ms moves the P99 of the
shortest RPCs by. The table prints the P50 and the P99 of the RPCs of at most 100 bytes and of
those under 1 KB.

In [ ]:
LEVELS = (8, 4, 2, 1)
pooled = {levels: {"100": [], "1k": [], "unanswered": 0, "drops": 0, "rpcs": 0} for levels in LEVELS}
for s in range(level_seeds):
    arrivals = poisson(hosts, load, duration_ns, seed + 100 * s)
    requests, warm = [], 0
    for levels in LEVELS:
        req, warm = request(hosts, arrivals, "levels-%d-%d" % (levels, s), levels=levels, cutoffs=CUTOFFS[levels])
        requests.append(req)
    for levels, answer in zip(LEVELS, sim(requests, "levels-seed-%d" % s)):
        small, unanswered = slowdowns(answer, arrivals, warm, 100)
        under, _ = slowdowns(answer, arrivals, warm, 999)
        pooled[levels]["100"] += small
        pooled[levels]["1k"] += under
        pooled[levels]["unanswered"] += unanswered
        pooled[levels]["drops"] += answer["drops"]
        pooled[levels]["rpcs"] += len(arrivals)
P99 = {}
print("%-7s %-7s %-12s %-12s %-12s %-12s %-10s %s" % (
    "levels", "RPCs", "100 B P50", "100 B P99", "<1 KB P50", "<1 KB P99", "unanswered", "drops"))
for levels in LEVELS:
    row = pooled[levels]
    P99[levels] = (fabric.percentile(row["100"], 99), fabric.percentile(row["1k"], 99))
    print("%-7d %-7d %-12.3f %-12.3f %-12.3f %-12.3f %-10d %d" % (
        levels, row["rpcs"], fabric.percentile(row["100"], 50), P99[levels][0],
        fabric.percentile(row["1k"], 50), P99[levels][1], row["unanswered"], row["drops"]))

## Load, the larger rack and the byte stream baseline

The same mix with 8 levels runs at every load of `loads` on `hosts` hosts and at `load` on
`big_hosts` hosts. The baseline is the same engine made a byte stream: one level, every transmit
and grant choice FIFO rather than SRPT (`fifo_every` 1, which the engine takes as the FIFO share
of every choice), and one message granted at a time, so a short message waits behind every byte
queued before it, as it would behind a TCP stream.

In [ ]:
LOAD_ROWS = []
load_requests = []
load_arrivals = []
for value in [float(x) for x in loads.split(",")]:
    arrivals = poisson(hosts, value, duration_ns, seed + 7)
    req, warm = request(hosts, arrivals, "load-%s" % value, cutoffs=CUTOFFS[8])
    load_requests.append(req)
    load_arrivals.append((value, hosts, arrivals, warm))
big = poisson(big_hosts, load, duration_ns // 2, seed + 11)
req, big_warm = request(big_hosts, big, "big", cutoffs=CUTOFFS[8])
load_requests.append(req)
load_arrivals.append((load, big_hosts, big, big_warm))
base_arrivals = poisson(hosts, load, duration_ns, seed)
req, base_warm = request(hosts, base_arrivals, "baseline", levels=1, cutoffs=CUTOFFS[1],
                         overrides={"fifo_every": 1, "max_overcommit": 1, "fifo_permille": 0})
load_requests.append(req)
load_arrivals.append(("baseline", hosts, base_arrivals, base_warm))
answers = sim(load_requests, "loads")
print("%-9s %-6s %-7s %-12s %-12s %-12s %-12s %-12s %-10s %s" % (
    "load", "hosts", "RPCs", "100 B P50", "100 B P99", "<1 KB P99", "all P50", "all P99", "unanswered",
    "downlinks busy"))
for (value, n, arrivals, warm), answer in zip(load_arrivals, answers):
    small, unanswered = slowdowns(answer, arrivals, warm, 100)
    under, _ = slowdowns(answer, arrivals, warm, 999)
    every, _ = slowdowns(answer, arrivals, warm, 10 ** 7)
    busy = answer["fabric"]["downlink_busy_permille"][:n]
    LOAD_ROWS.append({"load": value, "hosts": n, "rpcs": len(arrivals), "p99_100": fabric.percentile(small, 99),
                      "p99_1k": fabric.percentile(under, 99), "unanswered": unanswered, "drops": answer["drops"]})
    print("%-9s %-6d %-7d %-12.3f %-12.3f %-12.3f %-12.3f %-12.3f %-10d %.1f %%" % (
        value, n, len(arrivals), fabric.percentile(small, 50), fabric.percentile(small, 99),
        fabric.percentile(under, 99), fabric.percentile(every, 50), fabric.percentile(every, 99),
        unanswered, sum(busy) / len(busy) / 10))
BASELINE = LOAD_ROWS[-1]

## Incast: a fan out of 10 KB responses

Host 0 of `big_hosts` hosts sends `n` requests of 100 bytes at one instant, spread over the other
hosts, and each server answers with 10,000 bytes, so every response converges on the one downlink of
host 0. With incast control, a client with more than 8 RPCs outstanding marks each new request, and
a server answers a marked request with no more than 300 unscheduled bytes, which under all or
nothing sends the response of 10,000 bytes by `START_MSG` and grants. Without it every response
arrives unscheduled at once, and the shared buffer of the switch drops what does not fit.

In [ ]:
FAN = []
fan_requests = []
for n in [int(x) for x in fan_outs.split(",")]:
    rpcs = [[0, 1 + i % (big_hosts - 1), 2 * START_NS, 100, 10_000] for i in range(n)]
    for incast in (True, False):
        req, warm = request(big_hosts, rpcs, "fan-%d-%s" % (n, incast), config={
            "client_slots": 256, "server_slots": 256, "page_len": 16384, "pages": 512, "max_gaps": 16},
            extra={"incast": incast, "host_configs": [{"host": 0, "client_slots": 4200, "server_slots": 64,
                                                       "page_len": 16384, "pages": 4400, "max_gaps": 16}],
                   "measure_from_ns": 2 * START_NS, "max_time_ns": 60_000_000_000})
        fan_requests.append(req)
        FAN.append((n, incast, warm))
print("%-6s %-7s %-9s %-9s %-10s %-12s %-10s %-9s %s" % (
    "fan", "incast", "answered", "drops", "restarts", "last ms", "max queue", "marked", "resends"))
FAN_ROWS = {}
for (n, incast, warm), answer in zip(FAN, sim(fan_requests, "fan")):
    ended = answer["rpcs"][warm:]
    answered = sum(1 for r in ended if r[3] == 1)
    last = max((r[2] - r[0] for r in ended if r[3] == 1), default=0)
    FAN_ROWS[(n, incast)] = {"answered": answered, "drops": answer["drops"], "stopped": answer["stopped"]}
    print("%-6d %-7s %-9d %-9d %-10d %-12.3f %-10d %-9d %d" % (
        n, incast, answered, answer["drops"], answer["stats"]["restarts"], last / 1e6,
        answer["fabric"]["max_queue_bytes"], answer["stats"]["incast_marked"], answer["stats"]["resends"]))

## Overcommitment

Every host of `hosts` hosts offers `overcommit_load` of its link in messages of W4 alone, whose long
messages need grants, and every receiver grants to at most `max_overcommit` messages at once, one a
sender, each at most one round trip of bytes beyond what arrived, RTTbytes, as SIGCOMM 2018 sets the
grants. RTTbytes is the link rate times the time a full packet takes to its receiver and a GRANT
takes back on the idle fabric, from `fabric.py`. With one message granted at a time a downlink idles
while the GRANT of the next message travels and while the one granted sender is busy elsewhere, and
with 8 it stays busy. The cell prints the share of the time every downlink was busy between 3 ms and
the end of the arrivals, and their mean is the use of the receiver's downlink the exit number asks
for.

In [ ]:
RTT_NS = fabric.one_way_ns(1424, RATE, PROPAGATION) + 2 * fabric.wire_ns(fabric.GRANT, RATE) + 2 * PROPAGATION
RTT_BYTES = RTT_NS * RATE // 8_000_000_000
print("round trip of a full packet and a GRANT: %d ns, RTTbytes %d" % (RTT_NS, RTT_BYTES))
OVER = {}
w4 = poisson(hosts, overcommit_load, duration_ns, seed + 3, workload=["w4"])
over_requests = []
for degree in (8, 1):
    req, over_warm = request(hosts, w4, "overcommit-%d" % degree, cutoffs=CUTOFFS[8],
                             overrides={"max_overcommit": degree, "window": RTT_BYTES},
                             extra={"measure_from_ns": 3 * START_NS, "measure_to_ns": START_NS + duration_ns})
    over_requests.append(req)
for degree, answer in zip((8, 1), sim(over_requests, "overcommit")):
    busy = answer["window"]["downlink_busy_permille"][:hosts]
    unanswered = sum(1 for r in answer["rpcs"][over_warm:] if r[3] != 1)
    OVER[degree] = sum(busy) / len(busy) / 10
    print("max_overcommit %d: downlinks busy %.1f %% on average, least %.1f %%, most %.1f %%, unanswered %d, drops %d" % (
        degree, OVER[degree], min(busy) / 10, max(busy) / 10, unanswered, answer["drops"]))

## The grant decisions over the trace

Three runs of `trace_duration_ns` of arrivals are traced: the mix at `load` with 8 and with 2
levels, and the overcommitment run with one message granted at a time. `trace.fabric` checks every
decision of every grant scheduler: at most one message a sender granted at one decision outside
the FIFO share, the level of each decision equal to `homa_grant_priority` of its rank among the
messages granted at once and their count, the priority of every GRANT equal to the level of the
last decision for its message, and the bytes a host expects, granted and not received over every
message it takes, within `max_incoming` at every decision. A trace the cell builds, with one
decision granting two messages of one sender, one at the wrong level, one GRANT at another level
and one decision beyond `max_incoming`, shows that each check refuses what it is written for.

In [ ]:
TRACED = [
    ("8 levels", 8, None),
    ("2 levels", 2, None),
    ("max_overcommit 1", 8, {"max_overcommit": 1}),
]
TRACE_ROWS = {}
arrivals = poisson(hosts, load, trace_duration_ns, seed + 13)
for label, levels, overrides in TRACED:
    path = OUT / ("trace-%s.ndjson" % label.replace(" ", "-"))
    req, warm = request(hosts, arrivals, label, levels=levels, cutoffs=CUTOFFS[levels], overrides=overrides,
                        extra={"trace": str(path)})
    answer = sim([req], "trace-" + label.replace(" ", "-"))[0]
    rows = trace.read(path)
    got = trace.fabric(rows, levels, 400_000, 40_000, fabric.homa_grant_priority)
    got["unanswered"] = sum(1 for r in answer["rpcs"][warm:] if r[3] != 1)
    TRACE_ROWS[label] = got
    path.unlink()
    print("%s: rows %d, decisions %d of which the FIFO share %d, GRANTs %d, one sender twice at a decision %d,"
          " levels off homa_grant_priority %d, GRANTs off their decision %d, GRANTs with no decision %d,"
          " most expected %d, above max_incoming %d, unanswered %d" % (
              label, len(rows), got["decisions"], got["fifo"], got["grants"], got["sender_twice"],
              got["level_mismatch"], got["grant_level_mismatch"], got["grants_without_decision"],
              got["worst_expected"], got["expected_over"], got["unanswered"]))
    for key in ("sender_examples", "level_examples", "grant_examples", "expected_examples"):
        if got[key]:
            print("  ", key, got[key][:2])

BAD = [
    ["M", 10, 1, 100, 200000, 2], ["M", 10, 1, 102, 200000, 2],
    ["P", 11, 1, 100, 50000, 1, 0, 2, 133333, False, 2], ["P", 11, 1, 102, 50000, 2, 1, 2, 133333, False, 2],
    ["R", 11, 1, 104, 0, 1424, 1500000, 3],
    ["G", 12, 1, 100, 50000, 1, 2], ["G", 12, 1, 102, 50000, 1, 2],
    ["M", 13, 1, 106, 900000, 4],
    ["P", 14, 1, 106, 450000, 0, 0, 1, 200000, False, 4],
]
bad = trace.fabric(BAD, 8, 400_000, 40_000, fabric.homa_grant_priority)
print("a trace built to break each check:", {k: bad[k] for k in (
    "sender_twice", "level_mismatch", "grant_level_mismatch", "expected_over", "unscheduled_too_long")})

## Computed cutoffs against the defaults

Host 0 receives `cutoff_messages` requests of the mix at `load` of its downlink from the other 15
hosts, counts the unscheduled bytes of every message in its histogram of 32 buckets a fifth of an
octave wide below its 40,000 unscheduled bytes, and then computes its cutoffs from it with
`Endpoint::recompute_cutoffs`. The share of each unscheduled level is the bytes of the messages of
at most 40,000 bytes its cutoffs give it over an equal share, the rule of SIGCOMM 2018 section 3.4,
read by `fabric.level_shares` over the lengths host 0 received and over a fresh sample of 400,000
lengths of the same mix. The default cutoffs of HomaModule are printed beside, which were not made
for this mix.

In [ ]:
g = random.Random(seed ^ 0x29)
mix = fabric.Workload(MIX, seed + 17)
received = [mix.draw() for _ in range(cutoff_messages)]
wire = sum(fabric.message_wire_bytes(n) for n in received)
span = int(wire * 8 / (load * RATE) * 1e9)
times = sorted(g.uniform(0, span) for _ in received)
rpcs = [[1 + i % (hosts - 1), 0, START_NS + int(t), n, 0] for i, (t, n) in enumerate(zip(times, received))]
req, warm = request(hosts, rpcs, "cutoffs", cutoffs=CUTOFFS[8], config={
    "client_slots": 512, "server_slots": 512, "page_len": 65536, "pages": 1024, "max_gaps": 16, "size_buckets": 32},
    extra={"host_configs": [{"host": 0, "client_slots": 64, "server_slots": 4096, "page_len": 65536,
                             "pages": 8192, "max_gaps": 16, "size_buckets": 32}],
           "recompute_at_ns": START_NS + span + 5_000_000, "max_time_ns": 60_000_000_000})
answer = sim([req], "cutoffs")[0]
COMPUTED = answer["cutoffs"][0]
fresh = fabric.Workload(MIX, seed + 19)
FRESH = [fresh.draw() for _ in range(400_000)]
SHARES = {
    "computed, host 0's lengths": fabric.level_shares(COMPUTED, received, 8),
    "computed, a fresh sample": fabric.level_shares(COMPUTED, FRESH, 8),
    "defaults, a fresh sample": fabric.level_shares(fabric.DEFAULT_CUTOFFS, FRESH, 8),
}
unanswered = sum(1 for r in answer["rpcs"][warm:] if r[3] != 1)
print("messages received by host 0: %d, unanswered %d, span of the arrivals %.1f ms" % (
    len(received), unanswered, span / 1e6))
print("computed cutoffs:", COMPUTED)
print("default cutoffs :", fabric.DEFAULT_CUTOFFS)
print("equal share cutoffs of host 0's lengths, exact:", fabric.equal_share_cutoffs(received, 8))
for label, shares in SHARES.items():
    print("%-28s shares of levels 7 to 4: %s" % (label, ", ".join("%.3f" % v for v in shares)))
CUTOFF_WORST = max(abs(v - 1) for label, shares in SHARES.items() if label.startswith("computed") for v in shares)

## Verdict

In [ ]:
print("panics or crashes of the seam:", len(PANICS))
for panic in PANICS:
    print("  ", panic)
p8, p4, p2, p1 = (P99[levels] for levels in (8, 4, 2, 1))
print("P99 slowdown of 100 byte RPCs: 8 levels %.3f, 2 levels %.3f" % (p8[0], p2[0]))
print("P99 slowdown under 1 KB: 8 levels %.4f, 4 levels %.4f, 2 levels %.4f, 1 level %.4f, byte stream %.3f,"
      " baseline over 8 levels %.1f" % (p8[1], p4[1], p2[1], p1[1], BASELINE["p99_1k"], BASELINE["p99_1k"] / p8[1]))
print("fan out of %d with incast control: %d answered, %d drops; without: %d answered, %d drops" % (
    4000, FAN_ROWS[(4000, True)]["answered"], FAN_ROWS[(4000, True)]["drops"],
    FAN_ROWS[(4000, False)]["answered"], FAN_ROWS[(4000, False)]["drops"]))
print("downlinks busy: max_overcommit 8 %.1f %%, 1 %.1f %%" % (OVER[8], OVER[1]))
print("computed cutoffs, worst share off an equal share: %.3f" % CUTOFF_WORST)
assert not PANICS, PANICS
assert all(pooled[levels]["unanswered"] == 0 for levels in LEVELS), pooled
assert all(row["unanswered"] == 0 for row in LOAD_ROWS), LOAD_ROWS
assert p8[0] <= 3 and p2[0] <= 4, (p8, p2)
assert p8[1] <= p4[1] <= p2[1] <= p1[1], (p8, p4, p2, p1)
assert 5 * p8[1] <= BASELINE["p99_1k"], (p8, BASELINE)
assert all(FAN_ROWS[(n, True)]["answered"] == n for n in [int(x) for x in fan_outs.split(",")]), FAN_ROWS
assert FAN_ROWS[(4000, True)]["drops"] == 0 and FAN_ROWS[(4000, False)]["drops"] > 0, FAN_ROWS
assert OVER[8] >= 85 and OVER[1] < 70, OVER
for label, got in TRACE_ROWS.items():
    assert got["sender_twice"] == 0 and got["level_mismatch"] == 0, (label, got)
    assert got["grant_level_mismatch"] == 0 and got["grants_without_decision"] == 0, (label, got)
    assert got["expected_over"] == 0 and got["unscheduled_too_long"] == 0 and got["unanswered"] == 0, (label, got)
assert bad["sender_twice"] == 1 and bad["level_mismatch"] == 1 and bad["grant_level_mismatch"] == 1, bad
assert bad["expected_over"] >= 1 and bad["unscheduled_too_long"] == 1, bad
assert CUTOFF_WORST <= 0.10, SHARES
print()
print("VERDICT: at %d percent load on W1 and W4 the P99 slowdown of 100 byte RPCs %.2f with 8 levels and"
      " %.2f with 2, the P99 under 1 KB %.3f, %.3f, %.3f and %.3f with 8, 4, 2 and 1 levels and %.0f"
      " times lower with 8 than the byte stream baseline, a fan out of 4,000 complete with 0 drops"
      " with incast control and %d drops without, the downlinks %.1f percent busy with"
      " max_overcommit 8 and %.1f with 1, 0 grants to two messages of one sender, 0 decisions above"
      " max_incoming, every scheduled level that of homa_grant_priority, computed cutoffs within"
      " %.1f percent of an equal share, and 0 panics" % (
          100 * load, p8[0], p2[0], p8[1], p4[1], p2[1], p1[1], BASELINE["p99_1k"] / p8[1],
          FAN_ROWS[(4000, False)]["drops"], OVER[8], OVER[1], 100 * CUTOFF_WORST))